In [ ]:
#!/usr/bin/env python3
"""
Generate 1D Polynomial Network V2: Cubic-Stabilized with Strong Interactions

CHANGES FROM V1:
  1. Added Cubic Damping (-δ * x^3): Guarantees stability against quadratic blowup.
  2. Increased Triangle Coupling (β): From 0.125 -> 2.5 (Huge increase).
  3. Decreased Linear Decay (γ): The system is now stabilized by the cubic term.
"""

import numpy as np
import json
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════════════════════════
# PARAMETERS (TUNED FOR HIGH NON-LINEARITY)
# ═══════════════════════════════════════════════════════════════════════════

# Network structure (Same as before)
EDGES = [
    (0, 1), (0, 2), (0,3), (1, 2), (2, 3), (2,8),(2,9), (3, 4), (3, 5), (4, 5),
    (5, 6), (5,8), (5,9), (6, 7), (6,8), (7, 8), (7, 9), (8, 9),
]
TRIANGLES = [(0, 1, 2), (7, 8, 9), (3,4,5), (6,7,8), (2,8,9),] # Added (3,4,5) explicitly to logic
N_NODES = 10

# --- NEW DYNAMICS SETTINGS ---
# We want x to range roughly [-2, 2] so x^2 terms are significant (~4.0).

# Linear Decay (Reduced significantly, let the cubic term handle stability)
GAMMA_BASE = 0.1
GAMMA_SPREAD = 0.05

# Cubic Damping (The "Safety Net" - prevents explosion)
# dx = ... - delta * x^3
DELTA = 0.011

# Coupling Strengths
ALPHA = 2.5           # Linear coupling (Edge)
BETA  = 0.65           # Quadratic coupling (Triangle) - VERY STRONG

# Noise
NOISE_STD = 0.50       # Increased slightly to explore phase space

# Simulation
T_TOTAL = 30.0
FS = 256.0
DT = 1.0 / FS
T_TRANSIENT = 5.0

OUTPUT_CSV = 'polynomial_network_v2_high_interaction.csv'
OUTPUT_JSON = 'polynomial_ground_truth_v2.json'

print("=" * 70)
print("1D POLYNOMIAL NETWORK V2: CUBIC DAMPING + STRONG TRIANGLES")
print("=" * 70)
print(f"   dx_i/dt = -γ_i·x_i - δ·x_i³ + Σ α·x_j + Σ β·x_j·x_k + σ dW")
print(f"   Stability provided by Cubic term (-{DELTA} x^3)")
print(f"   Triangle Strength (β): {BETA} (Previously 0.125)")

# ═══════════════════════════════════════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════════════════════════════════════

# Adjacency
edge_pairs = {i: [] for i in range(N_NODES)}
for (i, j) in EDGES:
    edge_pairs[i].append(j); edge_pairs[j].append(i)

triangle_membership = {i: [] for i in range(N_NODES)}
for tri in TRIANGLES:
    for node in tri:
        others = [n for n in tri if n != node]
        triangle_membership[node].append(tuple(others))

# Random Seed
np.random.seed(101) # New seed

# Parameters
GAMMA_VEC = GAMMA_BASE + GAMMA_SPREAD * (2.0 * np.random.rand(N_NODES) - 1.0)

# Signed Edges
EDGE_COEFF = {}
for (i, j) in EDGES:
    sign_ij = np.random.choice([-1.0, 1.0])
    sign_ji = np.random.choice([-1.0, 1.0])
    EDGE_COEFF[(i, j)] = ALPHA * sign_ij
    EDGE_COEFF[(j, i)] = ALPHA * sign_ji

# Signed Triangles
TRI_COEFF = {i: {} for i in range(N_NODES)}
for tri in TRIANGLES:
    a, b, c = tri
    # Apply to all permutations
    for i, (j, k) in [(a, (b, c)), (b, (a, c)), (c, (a, b))]:
        sign = np.random.choice([-1.0, 1.0])
        TRI_COEFF[i][(j, k)] = BETA * sign

# ═══════════════════════════════════════════════════════════════════════════
# SIMULATION
# ═══════════════════════════════════════════════════════════════════════════

def dynamics(state):
    dstate = np.zeros_like(state)
    for i in range(N_NODES):
        x_i = state[i]

        # 1. Linear Decay
        dx = -GAMMA_VEC[i] * x_i

        # 2. Cubic Stability Term (NEW)
        dx += -DELTA * (x_i**3)

        # 3. Edge Coupling
        for j in edge_pairs[i]:
            dx += EDGE_COEFF[(i, j)] * state[j]

        # 4. Triangle Coupling
        for (j, k) in triangle_membership[i]:
            coeff = TRI_COEFF[i][(j, k)]
            dx += coeff * state[j] * state[k]

        dstate[i] = dx
    return dstate

print(f"\n🚀 Simulating...")
t_all = np.arange(0, T_TOTAL, DT)
state = np.random.randn(N_NODES) # Start random
traj = []

for t in t_all:
    traj.append(state)
    # Euler-Maruyama
    det_drift = dynamics(state)
    diffusion = NOISE_STD * np.sqrt(DT) * np.random.randn(N_NODES)
    state = state + det_drift * DT + diffusion

    # Safety Check
    if np.max(np.abs(state)) > 20:
        print("⚠ WARNING: Stability Instability Detected! (Cubic term too weak?)")
        break

traj = np.array(traj)
n_transient = int(T_TRANSIENT * FS)
data_output = traj[n_transient:, :].T # (Nodes, Samples)

print(f"   Max Amplitude: {np.max(np.abs(data_output)):.2f}")
print(f"   (Should be > 1.5 for quadratic terms to matter)")

# ═══════════════════════════════════════════════════════════════════════════
# QUICK CHECK OF CONTRIBUTIONS
# ═══════════════════════════════════════════════════════════════════════════

print("\n📊 Quick Force Analysis (Average Magnitude):")
X = data_output
avgs = []
for i in range(N_NODES):
    # Calculate term forces on the final dataset
    f_linear = -GAMMA_VEC[i] * X[i]
    f_cubic  = -DELTA * (X[i]**3)

    f_edge = np.zeros_like(X[i])
    for j in edge_pairs[i]:
        f_edge += EDGE_COEFF[(i,j)] * X[j]

    f_tri = np.zeros_like(X[i])
    for (j,k) in triangle_membership[i]:
        f_tri += TRI_COEFF[i][(j,k)] * X[j] * X[k]

    m_lin = np.mean(np.abs(f_linear))
    m_cub = np.mean(np.abs(f_cubic))
    m_edg = np.mean(np.abs(f_edge))
    m_tri = np.mean(np.abs(f_tri))

    total = m_lin + m_cub + m_edg + m_tri
    if total == 0: total = 1

    print(f"   Node {i}: Linear {m_lin/total*100:.0f}% | Cubic {m_cub/total*100:.0f}% | Edge {m_edg/total*100:.0f}% | Tri {m_tri/total*100:.0f}%")

# ═══════════════════════════════════════════════════════════════════════════
# SAVING
# ═══════════════════════════════════════════════════════════════════════════

np.savetxt(OUTPUT_CSV, data_output.T, delimiter=',', fmt='%.8f')

# Update Ground Truth Structure
gt = {
    'n_nodes': N_NODES,
    'parameters': {
        'model_type': 'cubic_stabilized_polynomial',
        'delta_cubic': DELTA,
        'gamma_vec': GAMMA_VEC.tolist(),
        'edge_coeff': {f"{i}-{j}": float(c) for (i, j), c in EDGE_COEFF.items()},
        'tri_coeff': {str(i): {f"{j}-{k}": float(c) for (j,k), c in TRI_COEFF[i].items()} for i in range(N_NODES)}
    },
    'simulation': {'dt': DT, 'noise_std': NOISE_STD}
}
with open(OUTPUT_JSON, 'w') as f:
    json.dump(gt, f, indent=2)

print(f"\n✓ Saved {OUTPUT_CSV}")

1D POLYNOMIAL NETWORK V2: CUBIC DAMPING + STRONG TRIANGLES
   dx_i/dt = -γ_i·x_i - δ·x_i³ + Σ α·x_j + Σ β·x_j·x_k + σ dW
   Stability provided by Cubic term (-0.011 x^3)
   Triangle Strength (β): 0.65 (Previously 0.125)

🚀 Simulating...
   Max Amplitude: 18.91
   (Should be > 1.5 for quadratic terms to matter)

📊 Quick Force Analysis (Average Magnitude):
   Node 0: Linear 2% | Cubic 8% | Edge 48% | Tri 42%
   Node 1: Linear 3% | Cubic 17% | Edge 51% | Tri 29%
   Node 2: Linear 0% | Cubic 4% | Edge 49% | Tri 47%
   Node 3: Linear 1% | Cubic 8% | Edge 49% | Tri 42%
   Node 4: Linear 1% | Cubic 5% | Edge 46% | Tri 48%
   Node 5: Linear 2% | Cubic 18% | Edge 49% | Tri 31%
   Node 6: Linear 0% | Cubic 3% | Edge 49% | Tri 47%
   Node 7: Linear 2% | Cubic 30% | Edge 21% | Tri 48%
   Node 8: Linear 1% | Cubic 9% | Edge 36% | Tri 54%
   Node 9: Linear 0% | Cubic 1% | Edge 48% | Tri 51%

✓ Saved polynomial_network_v2_high_interaction.csv


In [1]:
import numpy as np

def _to_numpy_array(A_matrix, gpu_available=False, cp_module=None):
    """
    Convert A_matrix to a NumPy array without modifying the original.
    """
    if gpu_available and cp_module is not None:
        return cp_module.asnumpy(A_matrix).copy()
    return np.array(A_matrix, copy=True)


def _get_triple_structure(A, maps0, i, j, k):
    """
    Internal helper for one unordered triple {i,j,k}.

    Returns:
        None if required rows are missing.

        Otherwise a dict containing:
          - row indices for linear and pair terms
          - undirected edge strengths e_ij, e_ik, e_jk
          - canonical triangle-driving magnitudes
          - common simplicial bound m_ijk
    """
    r_i  = maps0[1].get((i,))
    r_j  = maps0[1].get((j,))
    r_k  = maps0[1].get((k,))
    r_ij = maps0[2].get((i, j))
    r_ik = maps0[2].get((i, k))
    r_jk = maps0[2].get((j, k))

    if None in (r_i, r_j, r_k, r_ij, r_ik, r_jk):
        return None

    # Undirected edge strengths from linear cross-couplings
    e_ij = max(abs(A[r_j, i]), abs(A[r_i, j]))
    e_ik = max(abs(A[r_k, i]), abs(A[r_i, k]))
    e_jk = max(abs(A[r_k, j]), abs(A[r_j, k]))

    # Canonical triangle-driving pair coefficients
    # x_j x_k in equation i, x_i x_k in equation j, x_i x_j in equation k
    t_i = abs(A[r_jk, i])
    t_j = abs(A[r_ik, j])
    t_k = abs(A[r_ij, k])

    m_ijk = min(e_ij, e_ik, e_jk)

    return {
        "rows": {
            "r_i": r_i, "r_j": r_j, "r_k": r_k,
            "r_ij": r_ij, "r_ik": r_ik, "r_jk": r_jk,
        },
        "edges": {
            "e_ij": e_ij, "e_ik": e_ik, "e_jk": e_jk,
        },
        "triangle_drivers": {
            "t_i": t_i, "t_j": t_j, "t_k": t_k,
        },
        "bound": m_ijk,
    }


def enforce_simplicial_hierarchy_clean(A_matrix, maps0, gpu_available=False, cp_module=None):
    """
    Enforce simplicial hierarchy on the canonical triangle-driving coefficients.

    For each unordered triple {i,j,k}, define
        e_ij = max(|A[x_j row, i]|, |A[x_i row, j]|),
        e_ik = max(|A[x_k row, i]|, |A[x_i row, k]|),
        e_jk = max(|A[x_k row, j]|, |A[x_j row, k]|),
    and then
        m_ijk = min(e_ij, e_ik, e_jk).

    The canonical triangle-driving entries are clipped to this bound:
        |A[x_j x_k row, i]| <= m_ijk
        |A[x_i x_k row, j]| <= m_ijk
        |A[x_i x_j row, k]| <= m_ijk

    This matches the intended simplicial closure rule:
    a triangle interaction cannot exceed its weakest supporting edge.

    Returns:
        A matrix of the same array type family as the input if possible:
        - NumPy array on CPU
        - CuPy array on GPU if cp_module is supplied and gpu_available=True
    """
    if 1 not in maps0 or 2 not in maps0:
        return A_matrix.copy()

    A = _to_numpy_array(A_matrix, gpu_available=gpu_available, cp_module=cp_module)
    n = A.shape[1]

    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                info = _get_triple_structure(A, maps0, i, j, k)
                if info is None:
                    continue

                m_ijk = info["bound"]
                rows = info["rows"]

                # Preserve signs, clip magnitudes
                A[rows["r_jk"], i] = np.sign(A[rows["r_jk"], i]) * min(abs(A[rows["r_jk"], i]), m_ijk)
                A[rows["r_ik"], j] = np.sign(A[rows["r_ik"], j]) * min(abs(A[rows["r_ik"], j]), m_ijk)
                A[rows["r_ij"], k] = np.sign(A[rows["r_ij"], k]) * min(abs(A[rows["r_ij"], k]), m_ijk)

    if gpu_available and cp_module is not None:
        return cp_module.asarray(A)
    return A


def count_simplicial_hierarchy_violations(
    A_matrix,
    maps0,
    thr=1e-6,
    count_mode="triple",
    gpu_available=False,
    cp_module=None,
):
    """
    Count true coefficient-level violations of the simplicial hierarchy.

    A triple {i,j,k} violates the hierarchy if any canonical triangle-driving entry
    exceeds the allowed simplicial bound m_ijk by more than `thr`, where

        m_ijk = min(e_ij, e_ik, e_jk)

    and the edge strengths are computed from linear cross-couplings.

    Parameters
    ----------
    A_matrix : array-like
        Coefficient matrix Xi.
    maps0 : dict
        Maps produced by build_maps_from_patterns.
    thr : float
        Numerical tolerance.
    count_mode : {"triple", "entry"}
        - "triple": count each unordered triple at most once
        - "entry": count each offending canonical triangle-driving entry separately
    gpu_available : bool
        Whether CuPy/GPU is active.
    cp_module : module or None
        CuPy module if used.

    Returns
    -------
    int
        Number of violations.
    """
    if count_mode not in {"triple", "entry"}:
        raise ValueError("count_mode must be 'triple' or 'entry'")

    if 1 not in maps0 or 2 not in maps0:
        return 0

    A = _to_numpy_array(A_matrix, gpu_available=gpu_available, cp_module=cp_module)
    n = A.shape[1]
    violations = 0

    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                info = _get_triple_structure(A, maps0, i, j, k)
                if info is None:
                    continue

                rows = info["rows"]
                m_ijk = info["bound"]

                offenders = [
                    abs(A[rows["r_jk"], i]) > m_ijk + thr,
                    abs(A[rows["r_ik"], j]) > m_ijk + thr,
                    abs(A[rows["r_ij"], k]) > m_ijk + thr,
                ]

                if count_mode == "entry":
                    violations += sum(offenders)
                else:
                    violations += int(any(offenders))

    return violations


def check_extracted_complex_closure(scores, n_nodes, tau2_q=0.75, tau3_q=0.75, S3_key="S3_max"):
    """
    Threshold the readout scores into a binary 1-skeleton and triangle set, then
    check whether every extracted triangle has all three extracted edges.

    This is NOT a coefficient-level hierarchy test.
    It is only a consistency check on the thresholded extracted complex.

    Parameters
    ----------
    scores : dict
        Output of readout_scores_multi_mode(...), expected to contain:
          - "S2"        : n x n edge score matrix
          - S3_key dict : triangle score dict
    n_nodes : int
        Number of nodes.
    tau2_q : float
        Edge quantile threshold.
    tau3_q : float
        Triangle quantile threshold.
    S3_key : str
        Which triangle score dictionary to use, e.g. "S3_max", "S3_mean", "S3_geom".

    Returns
    -------
    result : dict
        {
            "tau2": edge threshold used,
            "tau3": triangle threshold used,
            "edges": set of frozenset({i,j}),
            "triangles": set of frozenset({i,j,k}),
            "closed_triangles": set of extracted triangles whose 3 edges are all present,
            "violating_triangles": set of extracted triangles missing at least one edge,
            "n_edges": int,
            "n_triangles": int,
            "n_closed_triangles": int,
            "n_violating_triangles": int,
        }
    """
    S2 = scores.get("S2", np.zeros((n_nodes, n_nodes)))
    S3_scores = scores.get(S3_key, {})

    # Threshold edges
    edge_vals = [S2[i, j] for i in range(n_nodes) for j in range(i + 1, n_nodes) if S2[i, j] > 0]
    if edge_vals:
        tau2 = float(np.quantile(edge_vals, tau2_q))
        edges = {
            frozenset((i, j))
            for i in range(n_nodes)
            for j in range(i + 1, n_nodes)
            if S2[i, j] >= tau2
        }
    else:
        tau2 = np.inf
        edges = set()

    # Threshold triangles
    tri_vals = list(S3_scores.values())
    if tri_vals:
        tau3 = float(np.quantile(tri_vals, tau3_q))
        triangles = {tri for tri, val in S3_scores.items() if val >= tau3}
    else:
        tau3 = np.inf
        triangles = set()

    closed_triangles = set()
    violating_triangles = set()

    for tri in triangles:
        i, j, k = sorted(tri)
        required_edges = {
            frozenset((i, j)),
            frozenset((i, k)),
            frozenset((j, k)),
        }
        if required_edges.issubset(edges):
            closed_triangles.add(tri)
        else:
            violating_triangles.add(tri)

    return {
        "tau2": tau2,
        "tau3": tau3,
        "edges": edges,
        "triangles": triangles,
        "closed_triangles": closed_triangles,
        "violating_triangles": violating_triangles,
        "n_edges": len(edges),
        "n_triangles": len(triangles),
        "n_closed_triangles": len(closed_triangles),
        "n_violating_triangles": len(violating_triangles),
    }

In [9]:
def extract_simplicial_complex_edge_first(scores, n_nodes, tau2, tau3, S3_key="S3_max"):
    """
    Edge-first simplicial extraction.

    1. Threshold S2 at tau2 to get edges.
    2. Find all triples whose 3 boundary edges are present.
    3. Among those edge-supported triples, keep only triangles with S3 >= tau3.

    Returns
    -------
    edges : set[frozenset]
    triangles : set[frozenset]
    candidate_triangles : set[frozenset]
        All edge-supported triangles before triangle thresholding.
    """
    S2 = scores.get("S2", np.zeros((n_nodes, n_nodes)))
    S3 = scores.get(S3_key, {})
    print(S3)

    # Step 1: threshold edges
    edges = {
        frozenset((i, j))
        for i in range(n_nodes)
        for j in range(i + 1, n_nodes)
        if S2[i, j] >= tau2
    }

    # Step 2: edge-supported triangle candidates
    candidate_triangles = set()
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            for k in range(j + 1, n_nodes):
                tri = frozenset((i, j, k))
                boundary = {
                    frozenset((i, j)),
                    frozenset((i, k)),
                    frozenset((j, k)),
                }
                if boundary.issubset(edges):
                    candidate_triangles.add(tri)

    # Step 3: threshold triangles only among supported triples
    triangles = {
        tri for tri in candidate_triangles
        if S3.get(tri, 0.0) >= tau3
    }

    return edges, triangles, candidate_triangles


def sweep_edge_first_thresholds(
    scores,
    gt_edges_set,
    gt_tris_set,
    n_nodes,
    q2s=np.linspace(0.30, 0.99, 15),
    q3s=np.linspace(0.10, 0.90, 19),
    S3_key="S3_max",
):
    """
    Sweep thresholds with edge-first simplicial extraction.

    For each q2:
      - compute tau2 from positive S2 values
      - threshold edges first
      - find edge-supported triangle candidates
      - sweep tau3 only over those supported triangles

    Returns a list of dicts containing:
      q2, q3, tau2, tau3,
      edge precision/recall/F1,
      triangle precision/recall/F1,
      number of edges / supported triples / kept triangles
    """
    S2 = scores.get("S2", np.zeros((n_nodes, n_nodes)))
    S3 = scores.get(S3_key, {})

    edge_vals = [S2[i, j] for i in range(n_nodes) for j in range(i + 1, n_nodes) if S2[i, j] > 0]
    if not edge_vals:
        return []

    out = []

    for q2 in q2s:
        tau2 = float(np.quantile(edge_vals, q2))

        # Threshold edges first
        edges = {
            frozenset((i, j))
            for i in range(n_nodes)
            for j in range(i + 1, n_nodes)
            if S2[i, j] >= tau2
        }

        # Edge metrics
        TP_e = len(edges & gt_edges_set)
        FP_e = len(edges - gt_edges_set)
        FN_e = len(gt_edges_set - edges)
        P_e = TP_e / (TP_e + FP_e) if (TP_e + FP_e) > 0 else 0.0
        R_e = TP_e / (TP_e + FN_e) if (TP_e + FN_e) > 0 else 0.0
        F1_e = 2 * P_e * R_e / (P_e + R_e) if (P_e + R_e) > 0 else 0.0

        # Supported triangles under this 1-skeleton
        supported_tris = []
        supported_scores = []

        for i in range(n_nodes):
            for j in range(i + 1, n_nodes):
                for k in range(j + 1, n_nodes):
                    tri = frozenset((i, j, k))
                    boundary = {
                        frozenset((i, j)),
                        frozenset((i, k)),
                        frozenset((j, k)),
                    }
                    if boundary.issubset(edges):
                        supported_tris.append(tri)
                        supported_scores.append(S3.get(tri, 0.0))

        # No supported triangles => record degenerate triangle stats
        if len(supported_tris) == 0:
            out.append({
                "q2": float(q2),
                "q3": None,
                "tau2": tau2,
                "tau3": np.inf,
                "edge_P": P_e,
                "edge_R": R_e,
                "edge_F1": F1_e,
                "tri_P": 0.0,
                "tri_R": 0.0,
                "tri_F1": 0.0,
                "n_edges": len(edges),
                "n_supported_tris": 0,
                "n_triangles": 0,
            })
            continue

        supported_scores = np.array(supported_scores, dtype=float)

        for q3 in q3s:
            tau3 = float(np.quantile(supported_scores, q3))

            pred_tris = {
                tri for tri, sc in zip(supported_tris, supported_scores)
                if sc >= tau3
            }

            TP_t = len(pred_tris & gt_tris_set)
            FP_t = len(pred_tris - gt_tris_set)
            FN_t = len(gt_tris_set - pred_tris)
            P_t = TP_t / (TP_t + FP_t) if (TP_t + FP_t) > 0 else 0.0
            R_t = TP_t / (TP_t + FN_t) if (TP_t + FN_t) > 0 else 0.0
            F1_t = 2 * P_t * R_t / (P_t + R_t) if (P_t + R_t) > 0 else 0.0

            out.append({
                "q2": float(q2),
                "q3": float(q3),
                "tau2": tau2,
                "tau3": tau3,
                "edge_P": P_e,
                "edge_R": R_e,
                "edge_F1": F1_e,
                "tri_P": P_t,
                "tri_R": R_t,
                "tri_F1": F1_t,
                "n_edges": len(edges),
                "n_supported_tris": len(supported_tris),
                "n_triangles": len(pred_tris),
            })

    return out


def print_edge_first_complex_from_result(res, n_nodes, S3_key="S3_max"):
    """
    Print the actual best edge-first simplicial complex stored in one result dict.
    Assumes run_for_scale returns:
      - scores
      - best_q2, best_q3, best_tau2, best_tau3
    """
    scores = res["scores"]
    tau2 = res["best_tau2"]
    tau3 = res["best_tau3"]

    edges, triangles, candidate_triangles = extract_simplicial_complex_edge_first(
        scores,
        n_nodes=n_nodes,
        tau2=tau2,
        tau3=tau3,
        S3_key=S3_key,
    )

    edges_sorted = sorted(tuple(sorted(e)) for e in edges)
    tris_sorted = sorted(tuple(sorted(t)) for t in triangles)
    cand_sorted = sorted(tuple(sorted(t)) for t in candidate_triangles)

    print("\n=== Best edge-first simplicial complex ===")
    print(f"rho         = {res['rho']}")
    print(f"scale       = {res['scale']}")
    print(f"S3 mode     = {S3_key}")
    print(f"best_q2     = {res['best_q2']}")
    print(f"best_q3     = {res['best_q3']}")
    print(f"best_tau2   = {res['best_tau2']}")
    print(f"best_tau3   = {res['best_tau3']}")
    print(f"edge_F1     = {res['edge_F1']:.3f}")
    print(f"tri_F1_max  = {res['tri_F1_max']:.3f}")
    print(f"viols_pre   = {res['viols_pre']}")
    print(f"viols_post  = {res['viols_post']}")

    print(f"\nEdges ({len(edges_sorted)}):")
    print(edges_sorted)

    print(f"\nEdge-supported triangle candidates ({len(cand_sorted)}):")
    print(cand_sorted)

    print(f"\nTriangles kept after triangle threshold ({len(tris_sorted)}):")
    print(tris_sorted)

In [ ]:
#!/usr/bin/env python3
# Full Simplicial SINDy pipeline + GRID SEARCH over SIMPL_LAMBDA_SCALE + SIMPL_RHO
# The grid search calls the SAME pipeline (ADMM + SOC + flooring + pruning + debiasing)
# for each lambda/rho combination.

import os
import math
import time
from itertools import combinations, product

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("✓ GPU (CuPy) detected")
    _ = cp.zeros(1)
except ImportError:
    GPU_AVAILABLE = False
    cp = np
    print("⚠️ GPU not available (using NumPy)")

xp = cp if GPU_AVAILABLE else np

# ----- Inline ground-truth (V2 Structure) -----
GT_EDGES_INLINE = {
    frozenset((0,1)), frozenset((0,2)), frozenset((0,3)), frozenset((1,2)),
    frozenset((2,3)), frozenset((2,8)), frozenset((2,9)),
    frozenset((3,4)), frozenset((3,5)), frozenset((4,5)),
    frozenset((5,6)), frozenset((5,8)), frozenset((5,9)),
    frozenset((6,7)), frozenset((7,8)), frozenset ((6,8)), frozenset((7,9)), frozenset((8,9)),
}
GT_TRIS_INLINE = {
    frozenset((0,1,2)),
    frozenset((3,4,5)), # Added for V2
    frozenset((7,8,9)), frozenset((6,7,8)), frozenset((2,8,9)),
}

# ---------- DIAGNOSTIC HELPERS ----------

def nnz_by_degree(Xi, maps, thr=1e-9):
    A = cp.asnumpy(Xi) if GPU_AVAILABLE else Xi
    nz = {"const": int(np.linalg.norm(A[0]) > thr)}
    nz["lin"]   = sum(int(np.linalg.norm(A[row]) > thr) for row in maps[1].values())
    nz["pairs"] = sum(int(np.linalg.norm(A[row]) > thr) for row in maps.get(2, {}).values())
    return nz

def gram_spectrum(G):
    Gh = cp.asnumpy(G) if GPU_AVAILABLE else G
    s = np.linalg.svd(Gh, compute_uv=False)
    return dict(smin=float(s[-1]), smax=float(s[0]), cond=float(s[0]/max(s[-1],1e-16)))

def kkt_grad_row_norms(G, B, Xi):
    R = G @ Xi - B
    return (cp.asnumpy(cp.linalg.norm(R, axis=1)) if GPU_AVAILABLE
            else np.linalg.norm(R, axis=1))

def sweep_triangle_thresholds(S3_scores, gt_tris_set, qs=np.linspace(0.5, 0.99, 11)):
    vals = np.array(list(S3_scores.values())) if S3_scores else np.array([])
    out = []
    for q in qs:
        if vals.size == 0:
            out.append({"q": float(q), "tau3": np.inf, "P":0,"R":0,"F1":0,"TP":0,"FP":0,"FN":len(gt_tris_set)})
            continue
        tau3 = float(np.quantile(vals, q))
        pred = {t for t, s in S3_scores.items() if s >= tau3}
        TP = len(pred & gt_tris_set)
        FP = len(pred - gt_tris_set)
        FN = len(gt_tris_set - pred)
        P  = TP / (TP + FP) if (TP+FP)>0 else 0.0
        R  = TP / (TP + FN) if (TP+FN)>0 else 0.0
        F1 = 2*P*R/(P+R) if (P+R)>0 else 0.0
        out.append({"q": float(q), "tau3": tau3, "P":P, "R":R, "F1":F1,
                    "TP":TP, "FP":FP, "FN":FN})
    return out

def per_degree_residuals(G, B, Xi, maps):
    A = Xi
    R_full = G @ A - B
    ls_full = 0.5 * float((R_full * R_full).sum())
    A_d2zero = A.copy()
    for row in maps.get(2, {}).values():
        A_d2zero[row, :] = 0.0
    R_d2zero = G @ A_d2zero - B
    ls_d2zero = 0.5 * float((R_d2zero * R_d2zero).sum())
    return dict(ls_full=ls_full, ls_no_pairs=ls_d2zero, delta_pairs=ls_d2zero - ls_full)

def enforce_simplicial_hierarchy(A, maps0, xp):
    """
    Forces rigorous simplicial closure over unordered triples {i,j,k}.
    The triangle-supporting pair coefficients (A_{jk,i}, A_{ik,j}, A_{ij,k})
    cannot exceed the minimum undirected edge score of the 1-skeleton (e_ij, e_ik, e_jk).
    """
    if 2 in maps0 and 1 in maps0:
        n = A.shape[1] # Number of nodes

        # Iterate over all possible unordered triples
        for i in range(n):
            for j in range(i + 1, n):
                for k in range(j + 1, n):

                    # 1. Get Linear Rows (for edges)
                    r_i = maps0[1].get((i,))
                    r_j = maps0[1].get((j,))
                    r_k = maps0[1].get((k,))
                    if None in (r_i, r_j, r_k): continue

                    # 2. Get Pair Rows (for triangle drivers)
                    r_jk = maps0[2].get(tuple(sorted((j, k))))
                    r_ik = maps0[2].get(tuple(sorted((i, k))))
                    r_ij = maps0[2].get(tuple(sorted((i, j))))
                    if None in (r_jk, r_ik, r_ij): continue

                    # 3. Compute Undirected Edge Scores (S2 logic)
                    # e_uv = max(|A_u,v|, |A_v,u|)
                    e_ij = xp.maximum(xp.abs(A[r_j, i]), xp.abs(A[r_i, j]))
                    e_ik = xp.maximum(xp.abs(A[r_k, i]), xp.abs(A[r_i, k]))
                    e_jk = xp.maximum(xp.abs(A[r_k, j]), xp.abs(A[r_j, k]))

                    # 4. The Global Simplicial Bound
                    m_ijk = xp.minimum(xp.minimum(e_ij, e_ik), e_jk)

                    # 5. Enforce on the Triangle Drivers
                    A[r_jk, i] = xp.sign(A[r_jk, i]) * xp.minimum(xp.abs(A[r_jk, i]), m_ijk)
                    A[r_ik, j] = xp.sign(A[r_ik, j]) * xp.minimum(xp.abs(A[r_ik, j]), m_ijk)
                    A[r_ij, k] = xp.sign(A[r_ij, k]) * xp.minimum(xp.abs(A[r_ij, k]), m_ijk)

    return A
def count_simplicial_violations(A_matrix, maps0, xp):
    """
    Counts how many unordered triples {i,j,k} have active triangle-driving
    pair coefficients despite lacking a complete 1-skeleton of bounding edges.
    """
    violations = 0
    A = cp.asnumpy(A_matrix) if GPU_AVAILABLE else A_matrix.copy()
    thr = 1e-6
    n = A.shape[1]

    if 2 in maps0 and 1 in maps0:
        for i in range(n):
            for j in range(i + 1, n):
                for k in range(j + 1, n):

                    r_i = maps0[1].get((i,))
                    r_j = maps0[1].get((j,))
                    r_k = maps0[1].get((k,))
                    r_jk = maps0[2].get(tuple(sorted((j, k))))
                    r_ik = maps0[2].get(tuple(sorted((i, k))))
                    r_ij = maps0[2].get(tuple(sorted((i, j))))

                    if None in (r_i, r_j, r_k, r_jk, r_ik, r_ij): continue

                    # Check if the triangle is attempting to form
                    tri_active = (abs(A[r_jk, i]) > thr or
                                  abs(A[r_ik, j]) > thr or
                                  abs(A[r_ij, k]) > thr)

                    if tri_active:
                        e_ij = max(abs(A[r_j, i]), abs(A[r_i, j]))
                        e_ik = max(abs(A[r_k, i]), abs(A[r_i, k]))
                        e_jk = max(abs(A[r_k, j]), abs(A[r_j, k]))

                        m_ijk = min(e_ij, e_ik, e_jk)

                        # If the bounding 1-skeleton is broken, it's a violation
                        if m_ijk <= thr:
                            violations += 1

    return violations
# ─────────────── PARAMETERS ───────────────────────────────
CSV_FILE = 'polynomial_network_v2_high_interaction.csv'

FS = 256.0
WIN_SG, ORDER = 29, 3
D_MAX = 2
# SIMPL_RHO will be swept, initial default below
SIMPL_RHO_DEFAULT = 0.95
ADMM_RHO = 3.0
ADMM_OVERRELAX = 1.6
MAX_ITERS = 500       # we'll use this inside run_for_scale
WIN_LEN = 5120
STRIDE = WIN_LEN
MAX_ROWS = 6000
R_TARGET_PC = 0.95
K_MIN = 600
K_MAX = 1000000
ROW_NORM_NNZ_THR = 1e-6
PARAM_ABS_NNZ_THR = 1e-8
OUTDIR = 'dyn_complexes_admm_gpu_FULL_GRID'
os.makedirs(OUTDIR, exist_ok=True)
PAIR_LAMBDA_MULT = 1.5

# ─────────────── INDEX PATTERNS / TOPOLOGY ─────────────────
def precompute_index_patterns(n, dmax, use_xp=True):
    out = {}
    if dmax >= 2 and n >= 2:
        iu0, iu1 = np.triu_indices(n, k=1)
        pairs = np.stack([iu0.astype(np.int32), iu1.astype(np.int32)], axis=1)
        out['pairs'] = cp.asarray(pairs) if (use_xp and GPU_AVAILABLE) else pairs
    if dmax >= 3 and n >= 3:
        triples = np.array(list(combinations(range(n), 3)), dtype=np.int32)
        out['triples'] = cp.asarray(triples) if (use_xp and GPU_AVAILABLE) else triples
    if dmax >= 4 and n >= 4:
        quads = np.array(list(combinations(range(n), 4)), dtype=np.int32)
        out['quads'] = cp.asarray(quads) if (use_xp and GPU_AVAILABLE) else quads
    return out

def build_maps_from_patterns(n, dmax, idx):
    maps = {1:{}, 2:{}, 3:{}, 4:{}}
    row = 0
    if dmax >= 1:
        for i in range(n):
            row += 1
            maps[1][(i,)] = row
    if dmax >= 2 and 'pairs' in idx:
        for a, b in (idx['pairs'].get() if GPU_AVAILABLE else idx['pairs']):
            row += 1
            maps[2][(int(a), int(b))] = row
    if dmax >= 3 and 'triples' in idx:
        for i, j, k in (idx['triples'].get() if GPU_AVAILABLE else idx['triples']):
            row += 1
            maps[3][(int(i), int(j), int(k))] = row
    if dmax >= 4 and 'quads' in idx:
        for i, j, k, l in (idx['quads'].get() if GPU_AVAILABLE else idx['quads']):
            row += 1
            maps[4][(int(i), int(j), int(k), int(l))] = row
    g_total = row + 1   # include constant row 0
    return maps, g_total

def build_soc_edges_from_maps(dmax, maps):
    edges = []
    if dmax >= 2:
        for (i,j), child in maps[2].items():
            edges += [(child, maps[1][(i,)]), (child, maps[1][(j,)])]
    if dmax >= 3:
        for (i,j,k), child in maps[3].items():
            for face in combinations((i,j,k), 2):
                edges.append((child, maps[2][tuple(sorted(face))]))
    if dmax >= 4:
        for (i,j,k,l), child in maps[4].items():
            for face in combinations((i,j,k,l), 3):
                edges.append((child, maps[3][tuple(sorted(face))]))
    return edges

def build_library_fast(XT, n, dmax, idx):
    T = XT.shape[0]
    parts = [xp.ones((T, 1), dtype=XT.dtype)]  # constant term
    if dmax >= 1:
        parts.append(XT)
    if dmax >= 2 and 'pairs' in idx:
        p = idx['pairs']
        parts.append(XT[:, p[:, 0]] * XT[:, p[:, 1]])
    if dmax >= 3 and 'triples' in idx:
        tr = idx['triples']
        parts.append(XT[:, tr[:,0]] * XT[:, tr[:,1]] * XT[:, tr[:,2]])
    if dmax >= 4 and 'quads' in idx:
        qd = idx['quads']
        parts.append(XT[:, qd[:,0]] * XT[:, qd[:,1]] * XT[:, qd[:,2]] * XT[:, qd[:,3]])
    return xp.concatenate(parts, axis=1)

# ─────────────── GRAM PRECOMPUTE ───────────────────────────
def precompute_gram(Theta_std, Y_scaled):
    T = max(1, Theta_std.shape[0])
    G = (Theta_std.T @ Theta_std) / T
    eps_ridge = 5e-3
    G = G + eps_ridge * xp.eye(G.shape[0], dtype=G.dtype)
    B = (Theta_std.T @ Y_scaled)  / T
    return G, B

# ─────────────── SOC PROJECTION (exact) + DYKSTRA ───────────────────────────
def project_rows_dykstra(Z, edges, alpha, iters=25, tol=1e-7):
    Zc = Z.copy()
    if not edges:
        return Zc
    ci = xp.asarray([e[0] for e in edges], dtype=xp.int32)
    pi = xp.asarray([e[1] for e in edges], dtype=xp.int32)
    E, ncols = ci.shape[0], Zc.shape[1]
    inc_c = xp.zeros((E, ncols), dtype=Zc.dtype)
    inc_p = xp.zeros((E, ncols), dtype=Zc.dtype)
    for _ in range(iters):
        Zold = Zc.copy()
        c_tmp = Zc[ci] - inc_c
        p_tmp = Zc[pi] - inc_p
        C = xp.linalg.norm(c_tmp, axis=1)
        P = xp.linalg.norm(p_tmp, axis=1)
        feas = (C <= alpha * P + 1e-12)
        if (~feas).any():
            idx = xp.where(~feas)[0]
            Ct, Pt = C[idx], P[idx]
            cti, pti = c_tmp[idx], p_tmp[idx]
            denom = 1.0 + alpha * alpha
            t = (alpha * Ct + Pt) / denom
            sc = xp.where(Ct > 0, (alpha * t) / Ct, 0.0)
            sp = xp.where(Pt > 0, t / Pt, 0.0)
            c_tmp[idx] = cti * sc[:, None]
            p_tmp[idx] = pti * sp[:, None]
        inc_c = c_tmp - (Zc[ci] - inc_c)
        inc_p = p_tmp - (Zc[pi] - inc_p)
        Zc[ci] = c_tmp
        Zc[pi] = p_tmp
        if float(xp.linalg.norm(Zc - Zold)) < tol:
            break
    return Zc

# ─────────────── DEGREE-AWARE WEIGHTS ───────────────────────────
def degree_weights(maps, g):
    w = xp.ones(g, dtype=xp.float32)
    w[0] = 0.0   # constant row
    for (_,), r in maps.get(1, {}).items():
        w[r] *= xp.array(1.0, dtype=xp.float32)
    for (_,_), r in maps.get(2, {}).items():
        w[r] *= xp.array(PAIR_LAMBDA_MULT, dtype=xp.float32)
    return w

# ─────────────── ADMM SOLVER ─────────────────
def solve_admm_gl_soc(
    G, B, edges, lamb, rho_hier, w=None,
    max_iters=300, rho_admm=3.0, Xi0=None,
    log_every=25, adaptive_rho=True, overrelax=1.6,
    trace_every=1, return_trace=True, freeze_cholesky=False,
    early_proj_every=1, late_proj_every=1,
    early_proj_iters=15, late_proj_iters=50,
    early_proj_tol=1e-7,  late_proj_tol=1e-8,
    dtype_for_solves=None,
):
    """
    ADMM with:
      - group-lasso penalty (row-wise ℓ2)
      - SOC hierarchy constraints via Dykstra projection on rows
    Returns:
      (Xi, C)                 if return_trace == False
      (Xi, C, trace_dict)     if return_trace == True
    """
    g, k = G.shape[0], B.shape[1]
    if w is None:
        w = xp.ones(g, dtype=G.dtype)

    # Optional dtype casting for stability
    if dtype_for_solves is not None and dtype_for_solves != G.dtype:
        Gs = G.astype(dtype_for_solves, copy=True)
        Bs = B.astype(dtype_for_solves, copy=True)
        Is = xp.eye(g, dtype=dtype_for_solves)
    else:
        Gs, Bs = G, B
        Is = xp.eye(g, dtype=G.dtype)

    # Variables
    Xi = xp.zeros((g, k), dtype=Gs.dtype) if Xi0 is None else Xi0.astype(Gs.dtype, copy=True)
    Z  = Xi.copy()
    C  = Xi.copy()
    Uz = xp.zeros_like(Xi)
    Uc = xp.zeros_like(Xi)

    # Linear system matrix
    A = Gs + (2.0 * rho_admm) * Is
    try:
        Lc = xp.linalg.cholesky(A)
        chol = True
    except Exception:
        chol = False

    # Should we actually record a trace?
    record_trace = (trace_every is not None) and (trace_every > 0) and return_trace

    trace = None
    if record_trace:
        trace = {
            "obj": [], "ls": [], "gl": [], "r_pri": [], "r_dual": [],
            "rho": [], "lam": [], "nnz_rows": [], "nnz_params": [],
            "soc_violations_before": [], "soc_violations_after": [], "cond_A": []
        }

    def count_soc_violations(M):
        if not edges:
            return 0
        ci = xp.asarray([e[0] for e in edges], dtype=xp.int32)
        pi = xp.asarray([e[1] for e in edges], dtype=xp.int32)
        cn = xp.linalg.norm(M[ci], axis=1)
        pn = xp.linalg.norm(M[pi], axis=1)
        return int((cn > rho_hier * pn + 1e-12).sum())

    def condition_number(A_mat):
        try:
            s = xp.linalg.svd(A_mat, compute_uv=False)
            return float(s[0] / max(s[-1], 1e-16))
        except Exception:
            return float("nan")

    if record_trace:
        trace["cond_A"].append(condition_number(A))

    lam = lamb

    for it in range(max_iters):
        Z_prev = Z.copy()
        C_prev = C.copy()

        # ---- Xi-step ----
        RHS = Bs + rho_admm * (Z - Uz + C - Uc)
        if chol and not freeze_cholesky:
            Y = xp.linalg.solve(Lc, RHS)
            Xi = xp.linalg.solve(Lc.T, Y)
        else:
            Xi = xp.linalg.solve(A, RHS)

        # Over-relaxation
        Xi_hat = overrelax * Xi + (1.0 - overrelax) * Z_prev

        # ---- Z-step: group lasso (row-wise shrinkage) ----
        Wz = Xi_hat + Uz
        norms = xp.sqrt((Wz * Wz).sum(axis=1) + 1e-16)
        shrink = xp.maximum(0.0, 1.0 - (lam * w) / (rho_admm * norms))
        Z = (Wz.T * shrink).T

        # ---- C-step: hierarchical SOC projection ----
        two_thirds = (2 * max_iters) // 3
        if it < two_thirds:
            proj_every = early_proj_every
            proj_iters = early_proj_iters
            proj_tol   = early_proj_tol
        else:
            proj_every = late_proj_every
            proj_iters = late_proj_iters
            proj_tol   = late_proj_tol

        soc_before = None
        if proj_every and (it % proj_every) == 0:
            if record_trace:
                soc_before = count_soc_violations(Xi_hat + Uc)
            C = project_rows_dykstra(Xi_hat + Uc, edges, rho_hier,
                                     iters=proj_iters, tol=proj_tol)
        else:
            C = Xi_hat + Uc

        soc_after = count_soc_violations(C) if record_trace else None

        # ---- Dual updates ----
        Uz += (Xi_hat - Z)
        Uc += (Xi_hat - C)

        # ---- Residuals ----
        r_pri = float(xp.linalg.norm(Xi - Z) + xp.linalg.norm(Xi - C))
        r_dual = float(rho_admm * xp.linalg.norm((Z - Z_prev) + (C - C_prev)))

        # ---- Trace bookkeeping ----
        if record_trace and (it % trace_every) == 0:
            R = Gs @ Xi - Bs
            ls = 0.5 * float(xp.sum(R * R))
            Z_row_norms = xp.sqrt((Z * Z).sum(axis=1) + 1e-16)
            gl = float(lam * xp.sum(w * Z_row_norms))

            trace["ls"].append(ls)
            trace["gl"].append(gl)
            trace["obj"].append(ls + gl)
            trace["r_pri"].append(r_pri)
            trace["r_dual"].append(r_dual)
            trace["rho"].append(rho_admm)
            trace["lam"].append(float(lam))
            nnz_rows = int((xp.linalg.norm(Z, axis=1) > ROW_NORM_NNZ_THR).sum())
            nnz_params = int((xp.abs(Z) > PARAM_ABS_NNZ_THR).sum())
            trace["nnz_rows"].append(nnz_rows)
            trace["nnz_params"].append(nnz_params)
            trace["soc_violations_before"].append(int(soc_before) if soc_before is not None else None)
            trace["soc_violations_after"].append(int(soc_after) if soc_after is not None else None)

        # ---- Optional logging (for the "full run") ----
        if log_every and ((it % log_every) == 0 or it == max_iters - 1):
            # If we didn't just compute ls/gl, recompute quickly
            if not (record_trace and trace_every and (it % trace_every) == 0):
                R = Gs @ Xi - Bs
                ls = 0.5 * float(xp.sum(R * R))
                gl = float(lam * xp.sum(w * xp.sqrt((Z * Z).sum(axis=1) + 1e-16)))
            nnz_rows = int((xp.linalg.norm(Z, axis=1) > ROW_NORM_NNZ_THR).sum())
            print(
                f"     ADMM it={it:3d} | ls={ls:.3e} gl={gl:.3e} "
                f"| obj={ls+gl:.3e} | r_pri={r_pri:.2e} r_dual={r_dual:.2e} ρ={rho_admm:.2g} "
                f"| nnz_rows={nnz_rows}"
            )

        # ---- Adaptive ρ ----
        if adaptive_rho and it % 5 == 0 and it > 20:
            ratio = r_pri / max(r_dual, 1e-12)
            if ratio > 1.8:
                rho_admm *= 1.25
                Uz /= 1.25
                Uc /= 1.25
            elif ratio < 0.55:
                rho_admm /= 1.25
                Uz *= 1.25
                Uc *= 1.25
            rho_admm = float(min(max(rho_admm, 2.0), 10.0))
            A = Gs + (2.0 * rho_admm) * Is
            if not freeze_cholesky:
                try:
                    Lc = xp.linalg.cholesky(A)
                    chol = True
                except Exception:
                    chol = False
            if record_trace:
                trace["cond_A"].append(condition_number(A))

        # ---- Convergence test ----
        if r_pri < 1e-3 and r_dual < 1e-3:
            break

    if return_trace:
        return Xi, C, trace
    else:
        return Xi, C



# ─────────────── READ-OUT (S2/S3/S4) ───────────────────────────
def readout_scores_multi_mode(Xi, n, dmax, maps):
    """
    Returns S2 (edges) and THREE variants of S3 (triangles):
      - S3_max:  max(w_i, w_j, w_k)  (Most lenient, best for recall)
      - S3_mean: mean(w_i, w_j, w_k) (Balanced)
      - S3_geom: geom(w_i, w_j, w_k) (Strict, requires symmetry)
    """
    A = cp.asnumpy(Xi) if GPU_AVAILABLE else Xi
    absA = np.abs(A)
    out = {}

    # ----- S2: edge scores (Unchanged, max is standard for edges) -----
    if dmax >= 1 and maps[1]:
        S2_dir = np.zeros((n, n))
        for j in range(n):
            row = maps[1][(j,)]
            for i in range(n):
                if i == j: continue
                S2_dir[i, j] = absA[row, i]
        out['S2'] = np.maximum(S2_dir, S2_dir.T)

    # ----- S3: Multi-mode triangle scores -----
    if dmax >= 2 and maps[2]:
        S3_max = {}
        S3_mean = {}
        S3_geom = {}

        pair_row = maps[2]
        for i in range(n):
            for j in range(i + 1, n):
                for k in range(j + 1, n):
                    tri = frozenset((i, j, k))

                    # Get row indices for pairs (j,k), (i,k), (i,j)
                    r_jk = pair_row.get(tuple(sorted((j, k))))
                    r_ik = pair_row.get(tuple(sorted((i, k))))
                    r_ij = pair_row.get(tuple(sorted((i, j))))

                    if (r_jk is None) or (r_ik is None) or (r_ij is None):
                        continue

                    # Extract weights from the three equations
                    # w_i is the coeff of x_j*x_k in equation i
                    w_i = absA[r_jk, i]
                    w_j = absA[r_ik, j]
                    w_k = absA[r_ij, k]

                    weights = [w_i, w_j, w_k]

                    # 1. Max (Lenient)
                    s_max = max(weights)
                    if s_max > 1e-9: S3_max[tri] = s_max

                    # 2. Arithmetic Mean (Balanced)
                    s_mean = sum(weights) / 3.0
                    if s_mean > 1e-9: S3_mean[tri] = s_mean

                    # 3. Geometric Mean (Strict - Current)
                    s_geom = (w_i * w_j * w_k) ** (1.0/3.0)
                    if s_geom > 1e-9: S3_geom[tri] = s_geom

        out['S3_max'] = S3_max
        out['S3_mean'] = S3_mean
        out['S3_geom'] = S3_geom

    return out

# ─────────────── EDGE THRESHOLD SWEEP ───────────────────────
def sweep_edge_thresholds(S2, gt_edges_set, qs=np.linspace(0.30, 0.99, 15)):
    nloc = S2.shape[0]
    vals = [S2[i, j] for i in range(nloc) for j in range(i+1, nloc) if S2[i, j] > 0]
    out = []
    if not vals:
        return out
    for q in qs:
        tau2 = float(np.quantile(vals, q))
        pred = {frozenset((i, j)) for i in range(nloc) for j in range(i+1, nloc) if S2[i, j] >= tau2}
        TP = len(pred & gt_edges_set)
        FP = len(pred - gt_edges_set)
        FN = len(gt_edges_set - pred)
        P  = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        R  = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        F1 = 2*P*R/(P+R) if (P+R) > 0 else 0.0
        out.append({"q": q, "tau2": tau2, "P": P, "R": R, "F1": F1})
    return out

# ─────────────── DATA LOADING & PREPROCESS ───────────────────────
print("\n📂 Loading data...")
if not os.path.exists(CSV_FILE):
    print(f"❌ Error: Data file '{CSV_FILE}' not found.")
    raise SystemExit(1)

raw = pd.read_csv(CSV_FILE, header=None).values.astype(np.float64)
# Drop degenerate channels
raw = raw[:, np.ptp(raw, axis=0) > 1e-12].T
n, T_raw = raw.shape
dt = 1 / FS
print(f"✓ {n} channels × {T_raw} samples ({T_raw/FS:.1f}s)")

half = (WIN_SG - 1) // 2
X_smooth  = savgol_filter(raw, WIN_SG, ORDER, axis=1, mode='interp')[:, half:-half]
dXdt_raw  = savgol_filter(raw, WIN_SG, ORDER, deriv=1, delta=dt, axis=1, mode='interp')[:, half:-half]
mu  = X_smooth.mean(axis=1, keepdims=True)
sig = X_smooth.std(axis=1, keepdims=True) + 1e-8
X = (X_smooth - mu) / sig
Y = dXdt_raw / sig

X_gpu = cp.asarray(X) if GPU_AVAILABLE else X
Y_gpu = cp.asarray(Y) if GPU_AVAILABLE else Y
print(f"✓ Data on {'GPU' if GPU_AVAILABLE else 'CPU'}")

# ─────────────── WINDOW SELECTION (single full window) ───────────────
print("\n🔍 Preparing single full-length window...")
def select_local(Xw):
    x0 = np.median(Xw, axis=1, keepdims=True)
    Xc = Xw - x0
    sigma = np.maximum(
        1e-8,
        1.4826 * np.median(
            np.abs(Xc - np.median(Xc, axis=1, keepdims=True)),
            axis=1, keepdims=True
        )
    )
    d_pc = np.sqrt(((Xc / sigma) ** 2).sum(axis=0)) / np.sqrt(Xc.shape[0])
    keep = np.where(d_pc <= R_TARGET_PC)[0]
    if keep.size < K_MIN:
        keep = np.argsort(d_pc)[:K_MIN]
    elif keep.size > K_MAX:
        keep = np.argsort(d_pc)[:K_MAX]
    return x0, np.sort(keep), d_pc

w_start = 0
w_end   = WIN_LEN
Xw_full = X[:, w_start:w_end]
x0_full, keep_full, d_pc_full = select_local(Xw_full)
metas = [{
    "w_start": w_start,
    "t_mid": (w_start + WIN_LEN / 2) * dt,
    "x0": x0_full.flatten(),
    "keep": keep_full
}]
print("✓ 1 window ready (entire dataset)")

# ─────────────── PRECOMPUTE TOPOLOGY ───────────────────────
print("\n⚙️  Precomputing topology (maps/edges) once...")
idx_patterns = precompute_index_patterns(n, D_MAX, use_xp=True)
maps0, g_total = build_maps_from_patterns(n, D_MAX, idx_patterns)
edges0 = build_soc_edges_from_maps(D_MAX, maps0)
print(f"✓ g_total={g_total} rows | edges={len(edges0)} (dmax={D_MAX})")

w_degree_base = degree_weights(maps0, g_total)

# ─────────────── BUILD LIBRARY, GRAM, STATS (once) ───────────────────────
# We use the single window meta
meta = metas[0]
w_start, w_end = meta["w_start"], meta["w_start"] + WIN_LEN

if GPU_AVAILABLE:
    Xw = X_gpu[:, w_start:w_end]
    Yw = Y_gpu[:, w_start:w_end]
    x0 = cp.asarray(meta["x0"].reshape(-1, 1))
    Xw_c = Xw - x0
    Xw_use = Xw_c[:, meta["keep"]]
    Yw_use = Yw[:, meta["keep"]]
    if Xw_use.shape[1] > MAX_ROWS:
        idx = cp.linspace(0, Xw_use.shape[1] - 1, MAX_ROWS, dtype=cp.int32)
        Xw_use = Xw_use[:, idx]
        Yw_use = Yw_use[:, idx]
    XT = Xw_use.T
    YT = Yw_use.T
else:
    Xw = X[:, w_start:w_end]
    Yw = Y[:, w_start:w_end]
    x0 = meta["x0"].reshape(-1, 1)
    Xw_c = Xw - x0
    Xw_use = Xw_c[:, meta["keep"]]
    Yw_use = Yw[:, meta["keep"]]
    if Xw_use.shape[1] > MAX_ROWS:
        idx = np.linspace(0, Xw_use.shape[1] - 1, MAX_ROWS, dtype=int)
        Xw_use = Xw_use[:, idx]
        Yw_use = Yw_use[:, idx]
    XT = Xw_use.T
    YT = Yw_use.T

Theta = build_library_fast(XT, n, D_MAX, idx_patterns)

Theta_mean = xp.mean(Theta, axis=0, keepdims=True)
Theta_stdv = xp.std(Theta, axis=0, keepdims=True) + 1e-8
Thetaz = (Theta - Theta_mean) / Theta_stdv

Y_mean = xp.mean(YT, axis=0, keepdims=True)
Y_stdv = xp.std(YT, axis=0, keepdims=True) + 1e-8
Y_scaled = (YT - Y_mean) / Y_stdv

Thetaz   = Thetaz.astype(xp.float32, copy=False)
Y_scaled = Y_scaled.astype(xp.float32, copy=False)
G, B = precompute_gram(Thetaz, Y_scaled)

# Stats needed for lambda
T_eff = Thetaz.shape[0]
G_eff = Thetaz.shape[1]

if GPU_AVAILABLE:
    med = cp.median(Y_scaled, axis=0, keepdims=True)
    mad = cp.median(cp.abs(Y_scaled - med), axis=0, keepdims=True)
    res_std = float(1.4826 * cp.mean(mad))
else:
    med = np.median(Y_scaled, axis=0, keepdims=True)
    mad = np.median(np.abs(Y_scaled - med), axis=0, keepdims=True)
    res_std = 1.4826 * float(np.mean(mad))

# Fix 2: per-row weights from feature std
col_std = xp.std(Thetaz, axis=0).astype(xp.float32)
w_rows = xp.ones(G.shape[0], dtype=xp.float32)
bad = (col_std < 1e-6)
if (cp.any(bad).item() if GPU_AVAILABLE else bool(np.any(bad))):
    w_rows = w_rows.copy()
    w_rows[bad] = xp.array(3.0, dtype=w_rows.dtype)

gt_edges_set = GT_EDGES_INLINE
gt_tris_set  = GT_TRIS_INLINE

# ─────────────── SINGLE-SCALE FULL PIPELINE WRAPPER ───────────────────────
def run_for_scale(scale, rho_val, verbose=False):
    """
    Run the FULL pipeline (ADMM + SOC + flooring + per-node pruning + debiasing)
    for a given lambda scale and rho value. Returns edge/tirangle P/R/F1 plus some diagnostics.
    """
    # 1. Compute lambda
    lam = scale * (res_std / max(1, math.sqrt(T_eff))) * math.sqrt(2.0 * math.log(max(2, G_eff)))
    # 2. ADMM solve
    w = (w_degree_base.astype(xp.float32) * w_rows).astype(xp.float32)
    # Using rho_val passed as argument
    Xi_gpu, C_gpu = solve_admm_gl_soc(
        G, B, edges0, lamb=lam, rho_hier=rho_val,
        w=w, max_iters=MAX_ITERS, rho_admm=ADMM_RHO,
        Xi0=None, log_every=0, adaptive_rho=True, overrelax=ADMM_OVERRELAX,
        trace_every=0, return_trace=False
    )

    # 3. Final SOC projection (Using rho_val)
    Xi_gpu = project_rows_dykstra(C_gpu, edges0, rho_val, iters=200, tol=1e-9)
    # 4. SURGICAL FLOORING (same as main script)
    A_main = cp.asnumpy(Xi_gpu) if GPU_AVAILABLE else Xi_gpu
    row_norms = np.linalg.norm(A_main, axis=1)
    w_host = cp.asnumpy(w) if GPU_AVAILABLE else w
    tau_vec = (float(lam) * w_host) / float(ADMM_RHO)

    has_active_child = np.zeros(len(row_norms), dtype=bool)
    if 3 in maps0 and maps0[3]:
        for (i, j, k), r_tri in maps0[3].items():
            if row_norms[r_tri] > ROW_NORM_NNZ_THR:
                for (a, b) in [(i,j), (i,k), (j,k)]:
                    edge_key = tuple(sorted((a, b)))
                    r_edge = maps0[2].get(edge_key)
                    if r_edge is not None:
                        has_active_child[r_edge] = True
    if 2 in maps0 and maps0[2]:
        for (i, j), r_edge in maps0[2].items():
            if row_norms[r_edge] > ROW_NORM_NNZ_THR:
                has_active_child[maps0[1][(i,)]] = True
                has_active_child[maps0[1][(j,)]] = True

    row_is_lin = np.zeros_like(row_norms, dtype=bool)
    for r in maps0.get(1, {}).values():
        row_is_lin[r] = True
    row_is_pair = np.zeros_like(row_norms, dtype=bool)
    for r in maps0.get(2, {}).values():
        row_is_pair[r] = True

    grad = kkt_grad_row_norms(G, B, Xi_gpu)

    cand = np.zeros_like(row_norms, dtype=bool)
    cand[row_is_lin]  |= (row_norms[row_is_lin]  <= 0.8 * tau_vec[row_is_lin])
    cand[row_is_pair] |= (row_norms[row_is_pair] <= 0.7 * tau_vec[row_is_pair])

    pair_parent_active = np.zeros_like(row_norms, dtype=bool)
    # (Parent-protection for pairs is currently disabled in your code.)

    safe = grad <= (float(lam) * w_host + 1e-9)
    mask = cand & safe & (~has_active_child) & (~pair_parent_active)

    if np.any(mask):
        if GPU_AVAILABLE:
            Xi_gpu[cp.asarray(mask), :] = 0.0
        else:
            Xi_gpu[mask, :] = 0.0

    # 5. PER-NODE TOP-K PRUNING (Top-4, 25% threshold) — same logic as your latest variant
   # pair_row_indices = sorted(list(maps0.get(2, {}).values()))

    #if pair_row_indices:
      #   A_check = cp.asnumpy(Xi_gpu) if GPU_AVAILABLE else Xi_gpu
      #   for target in range(n):
       #     col_pairs = np.abs(A_check[pair_row_indices, target])
        #    if np.max(col_pairs) < 1e-9:
         #       continue

          #   local_max = np.max(col_pairs)
           # local_thr = 0.25 * local_max

            # Top-4 by magnitude
            #sort_idx = np.argsort(col_pairs)[::-1]
            #top_k_indices = sort_idx[:4]

            #kth_val = col_pairs[top_k_indices[-1]]
            #effective_thr = max(local_thr, kth_val * 0.99)

            #for local_idx, val in enumerate(col_pairs):
             #   if val < effective_thr:
              #      global_row = pair_row_indices[local_idx]
               #     if GPU_AVAILABLE:
                #        Xi_gpu[global_row, target] = 0.0
                 #   else:
                  #      Xi_gpu[global_row, target] = 0.0

    # 6. DEBIASING: refit on support using Gram
   # ---------------------------------------------------------
    # ---------------------------------------------------------
    # ---> NEW: Count violations BEFORE debiasing
    viols_before = count_simplicial_hierarchy_violations(
        Xi_gpu,
        maps0,
        thr=1e-6,
        count_mode="triple",
        gpu_available=GPU_AVAILABLE,
        cp_module=cp if GPU_AVAILABLE else None,
    )



    # ---------------------------------------------------------
    # 6. CONSTRAINED DEBIASING (Projected Gradient Descent)
    # ---------------------------------------------------------
    A_check = cp.asnumpy(Xi_gpu) if GPU_AVAILABLE else Xi_gpu
    active_mask = (np.linalg.norm(A_check, axis=1) > ROW_NORM_NNZ_THR)

    if np.any(active_mask):
        Xi_deb = Xi_gpu.copy()

        if GPU_AVAILABLE:
            active_idx = cp.where(cp.asarray(active_mask))[0]
            G_sub = G[cp.ix_(active_idx, active_idx)]
            L = float(cp.linalg.eigvalsh(G_sub)[-1])
        else:
            active_idx = np.where(active_mask)[0]
            G_sub = G[np.ix_(active_idx, active_idx)]
            L = float(np.linalg.eigvalsh(G_sub)[-1])

        alpha = 1.0 / (L + 1e-5)

        for _ in range(300):
            Grad = G @ Xi_deb - B
            Xi_deb = Xi_deb - alpha * Grad

            if GPU_AVAILABLE:
                Xi_deb[cp.asarray(~active_mask), :] = 0.0
            else:
                Xi_deb[~active_mask, :] = 0.0

            Xi_deb = enforce_simplicial_hierarchy_clean(
                Xi_deb,
                maps0,
                gpu_available=GPU_AVAILABLE,
                cp_module=cp if GPU_AVAILABLE else None,
            )


        Xi_gpu = Xi_deb

    # ---> NEW: Count violations AFTER debiasing
    viols_after = count_simplicial_hierarchy_violations(
        Xi_gpu,
        maps0,
        thr=1e-6,
        count_mode="triple",
        gpu_available=GPU_AVAILABLE,
        cp_module=cp if GPU_AVAILABLE else None,
    )


    # 7. DIAGNOSTICS (optional)
    Xi_final = Xi_gpu

    # Get all 3 variants
    scores = readout_scores_multi_mode(Xi_final, n, D_MAX, maps0)
    S2 = scores.get('S2', np.zeros((n, n)))

        # --- EDGE-FIRST SIMPLICIAL SWEEP (using S3_max) ---
    joint_sweep = sweep_edge_first_thresholds(
        scores,
        gt_edges_set,
        gt_tris_set,
        n_nodes=n,
        q2s=np.linspace(0.30, 0.99, 15),
        q3s=np.linspace(0.10, 0.90, 19),
        S3_key="S3_max",
    )

    if joint_sweep:
        # Prioritize triangle F1, then edge F1, then fewer simplices
        best_joint = max(
            joint_sweep,
            key=lambda z: (
                z["tri_F1"],
                z["edge_F1"],
                -z["n_triangles"],
                -z["n_edges"],
            )
        )

        F1_e = best_joint["edge_F1"]
        F1_t_max = best_joint["tri_F1"]
        best_q2 = best_joint["q2"]
        best_q3 = best_joint["q3"]
        best_tau2 = best_joint["tau2"]
        best_tau3 = best_joint["tau3"]
    else:
        F1_e = 0.0
        F1_t_max = 0.0
        best_q2 = None
        best_q3 = None
        best_tau2 = np.inf
        best_tau3 = np.inf
    return {

        "scores": scores,
        "scale": scale,
        "rho": rho_val,
        "edge_F1": F1_e,
        "tri_F1_max": F1_t_max,
        "best_q2": best_q2,
        "best_q3": best_q3,
        "best_tau2": best_tau2,
        "best_tau3": best_tau3,
        "viols_pre": viols_before,
        "viols_post": viols_after,
    }

# ─────────────── GRID SEARCH ───────────────────────
print("\n🚀 Starting MULTI-METRIC lambda-scale + RHO grid search...")

LAMBDA_SCALES = np.linspace(1.0, 1.0, 1)
RHO_SCALES = np.linspace(0.75, 1.1, 1)

all_results = []

# Update the header
print(f"{'Rho':<6} | {'Scale':<6} | {'EdgeF1':<6} | {'TriF1':<6} | {'Best q2':<8} | {'Best q3':<8} | {'Viols_Pre':<9} | {'Viols_Post':<10}")
print("-" * 95)

for rho in RHO_SCALES:
    for scale in LAMBDA_SCALES:
        res = run_for_scale(scale, rho, verbose=False)
        all_results.append(res)

        # Update the row printout
        print(f"{res['rho']:<6.3f} | {res['scale']:<6.3f} | {res['edge_F1']:<6.3f} | "
        f"{res['tri_F1_max']:<6.3f} | {str(res['best_q2']):<8} | {str(res['best_q3']):<8} | "
        f"{res['viols_pre']:<9d} | {res['viols_post']:<10d}")

df = pd.DataFrame(all_results)
out_csv = os.path.join(OUTDIR, "grid_search_full_pipeline.csv")
df.to_csv(out_csv, index=False)
print(f"\n✅ Saved full-pipeline grid search results to: {out_csv}")

# Update the final summary print
print(df[['rho', 'scale', 'edge_F1', 'tri_F1_max', 'best_q2', 'best_q3', 'viols_pre', 'viols_post']].head())

✓ GPU (CuPy) detected

📂 Loading data...
✓ 10 channels × 6400 samples (25.0s)
✓ Data on GPU

🔍 Preparing single full-length window...
✓ 1 window ready (entire dataset)

⚙️  Precomputing topology (maps/edges) once...
✓ g_total=56 rows | edges=90 (dmax=2)

🚀 Starting MULTI-METRIC lambda-scale + RHO grid search...
Rho    | Scale  | EdgeF1 | TriF1  | Best q2  | Best q3  | Viols_Pre | Viols_Post
-----------------------------------------------------------------------------------------------
edges: 33, triangles: 87. Correct Triangles: 36, Violations: 51
0.750  | 1.000  | 0.882  | 0.750  | 0.645    | 0.5888888888888889 | 4         | 0         

✅ Saved full-pipeline grid search results to: dyn_complexes_admm_gpu_FULL_GRID/grid_search_full_pipeline.csv
    rho  scale   edge_F1  tri_F1_max  best_q2   best_q3  viols_pre  viols_post
0  0.75    1.0  0.882353        0.75    0.645  0.588889          4           0


In [11]:
# Print the actual best edge-first simplicial complex for the first/current run
if all_results:
    print_edge_first_complex_from_result(all_results[0], n_nodes=n, S3_key="S3_max")

{frozenset({0, 1, 2}): np.float64(0.1633989242330309), frozenset({0, 1, 3}): np.float64(0.10082217567727979), frozenset({0, 1, 6}): np.float64(0.01721132930158283), frozenset({0, 1, 7}): np.float64(0.07764974428518646), frozenset({0, 1, 8}): np.float64(0.012513724131315159), frozenset({0, 1, 9}): np.float64(0.030183132211290698), frozenset({0, 2, 3}): np.float64(0.051565918024042645), frozenset({0, 2, 4}): np.float64(0.016358866211381957), frozenset({0, 2, 5}): np.float64(0.02547188658718186), frozenset({0, 2, 6}): np.float64(0.012975963730065489), frozenset({0, 2, 7}): np.float64(0.016555451822286876), frozenset({0, 8, 2}): np.float64(0.021177977313066816), frozenset({0, 9, 2}): np.float64(0.018694264450823315), frozenset({0, 3, 4}): np.float64(0.06851837876022732), frozenset({0, 3, 5}): np.float64(0.037268870241215425), frozenset({0, 3, 6}): np.float64(0.03279676772651202), frozenset({0, 3, 7}): np.float64(0.07199730663577819), frozenset({0, 8, 3}): np.float64(0.012976414634965643), 